# Tune `linear_lr`

Linear (RF top-k + T² + hub×hub interactions) + elastic-net logistic (saga). Repeated stratified CV on the train split;
writes [`data/processed/tuned/linear_lr.json`](../data/processed/tuned/linear_lr.json).

Tune all models: [`tune_all.ipynb`](tune_all.ipynb).


**Classifier:** `elastic_net_lr` (saga + `penalty="elasticnet"`) in `scripts/secom_pipelines.py`.

**Stage 1 (hyperparameters):** RF top-k (`top_k`), hub count (`n_hubs`), classifier `C`, `l1_ratio`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

Shared sensor branch adds an **isolation forest** `decision_function` score after T² and hub interactions (unsupervised, refit per CV fold). Grid includes `isolation_forest__n_estimators`.

After hubs: **neighbor_fail_rate** (RF-weighted kNN mean train-label rate; tune `neighbor_fail_rate__n_neighbors`), then isolation forest.


In [7]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold_profiles,
    tuned_params_path,
)

MODEL_ID = "linear_lr"
spec = MODEL_SPECS[MODEL_ID]


In [8]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [9]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_branch__select_t2_hubs__top_k,preprocess__sensor_branch__select_t2_hubs__n_hubs,preprocess__sensor_branch__cluster__smart_corr__threshold,classifier__estimator__C,classifier__estimator__l1_ratio
0,[35],NaN,NaN,NaN,NaN
1,NaN,[5],NaN,NaN,NaN
2,NaN,NaN,[0.75],NaN,NaN
3,NaN,NaN,NaN,"[0.005, 0.0075]",NaN
4,NaN,NaN,NaN,NaN,"[0.3, 0.4, 0.5]"


In [10]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


linear_lr: 6 candidates x 10 folds = 60 fits


GridSearchCV 60 fits:   0%|          | 0/60 [00:00<?, ?it/s]

  0%|          | 0/60 [00:00<?, ?it/s]

Fitting 10 folds for each of 6 candidates, totalling 60 fits


In [11]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,top_k,n_hubs,corr_threshold,c,l1_ratio,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
4,35,5,0.75,0.0075,0.4,50.175968,0.361895,0.498240,0.588235,1.764706,99.059829,2.202410,0.726620,0.036837,0.165800,0.039889
3,35,5,0.75,0.0075,0.3,50.128205,0.319800,0.498718,0.000000,0.000000,99.743590,0.639600,0.727311,0.043429,0.164967,0.036315
1,35,5,0.75,0.0050,0.4,50.069130,0.351543,0.499309,0.588235,1.764706,99.273504,1.599570,0.706889,0.042724,0.163128,0.040646
0,35,5,0.75,0.0050,0.3,50.154600,0.336235,0.498454,0.588235,1.764706,99.102564,2.080016,0.714991,0.048411,0.160914,0.031305
5,35,5,0.75,0.0075,0.5,49.983660,0.515367,0.500163,0.588235,1.764706,99.444444,1.147498,0.708209,0.045226,0.155623,0.020855
2,35,5,0.75,0.0050,0.5,50.111865,0.319189,0.498881,0.588235,1.764706,99.188034,1.837607,0.690964,0.042580,0.144840,0.020691


In [ ]:
threshold_result = tune_classifier_threshold_profiles(spec, X_train, y_train, cv_summary)
print("Stage 2 — F-beta thresholds (F1 conservative / F2 neutral / F3 aggressive):")
for pid, prof in threshold_result["profiles"].items():
    print(
        f"  {pid}: threshold={prof['best_threshold']:.4f}, "
        f"mean_fbeta={prof['mean_fbeta']:.4f}, "
        f"mean_ber={prof['mean_ber_percent']:.2f}%"
    )
print(
    f"Deploy (F2): threshold={threshold_result['best_threshold']:.4f}, "
    f"mean_fbeta={threshold_result['mean_fbeta']:.4f}"
)
display(threshold_result["objective_curves"].head(10))
if "per_threshold_mean_ber" in threshold_result:
    display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/10 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/linear_lr.json


{'preprocess__sensor_branch__select_t2_hubs__top_k': 35,
 'preprocess__sensor_branch__select_t2_hubs__n_hubs': 5,
 'preprocess__sensor_branch__cluster__smart_corr__threshold': 0.75,
 'preprocess__sensor_branch__select_t2_hubs__neighbor_n_neighbors': 40,
 'classifier__C': 0.007,
 'classifier__l1_ratio': 0.3}